# 02 — Spatial scoring and cell-type composition

This notebook scores each spatial spot against the reference-derived cell-type signatures and converts those scores into relative cell-type composition estimates.

In [1]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src.analysis import (
    GENE_COLUMNS,
    compute_celltype_signatures,
    load_reference,
    load_spatial_expression,
    normalise_scores_to_composition,
    score_spots_against_signatures,
)
from src.plotting import save_celltype_composition_heatmap

## Conceptual setup

- Each spatial spot is represented as a vector of gene expression values.
- We compare each spot to each cell-type signature using a similarity measure.

In this implementation, we use cosine similarity as a simple proxy for how well a spot matches a given cell type.

In [5]:
spatial_df = load_spatial_expression("../data/raw/spatial_expression.csv")
signatures_df = pd.read_csv("../data/output/celltype_signatures.csv")

spatial_df.head(), signatures_df.head()

(  spot_id  GeneA  GeneB  GeneC  GeneD  GeneE
 0      s1      7      2      2      1      2
 1      s2      2      8      3      2      7
 2      s3      2      1      8      7      1
 3      s4      3      3      5      4      6
 4      s5      6      3      2      1      2,
     cell_type     GeneA     GeneB     GeneC     GeneD     GeneE
 0  Epithelial  1.666667  0.666667  8.000000  7.333333  1.333333
 1     Myeloid  0.666667  8.000000  3.333333  1.666667  7.333333
 2     Stromal  2.666667  2.333333  4.666667  4.666667  7.666667
 3      T_cell  8.000000  2.333333  0.666667  0.333333  1.666667)

## Spot-to-cell-type scoring

Each row corresponds to a spatial spot.

Each column corresponds to a cell type.

Values represent similarity between the spot expression profile and the cell-type signature.

Higher values indicate stronger similarity.

In [3]:
scores_df = score_spots_against_signatures(spatial_df, signatures_df)
scores_df

,spot_id,Epithelial,Myeloid,Stromal,T_cell
0,s1,0.447001,0.482486,0.624024,0.977574
1,s2,0.448535,0.991978,0.826439,0.503762
2,s3,0.998284,0.438651,0.729804,0.297293
3,s4,0.780868,0.833116,0.985908,0.549210
4,s5,0.466688,0.603815,0.664420,0.957095
5,s6,0.564617,0.987046,0.882503,0.417557
6,s7,0.983533,0.577137,0.814655,0.364990
7,s8,0.704461,0.851088,0.993915,0.549781


In [6]:
composition_df = normalise_scores_to_composition(scores_df)
composition_df

,spot_id,Epithelial,Myeloid,Stromal,T_cell
0,s1,0.176605,0.190624,0.246544,0.386227
1,s2,0.161884,0.358023,0.298277,0.181817
2,s3,0.405142,0.178022,0.296183,0.120653
3,s4,0.247965,0.264557,0.313076,0.174402
4,s5,0.173360,0.224298,0.246811,0.355531
5,s6,0.197992,0.346123,0.309463,0.146423
6,s7,0.358912,0.210610,0.297285,0.133193
7,s8,0.227301,0.274611,0.320696,0.177392


In [7]:
Path("../data/output").mkdir(parents=True, exist_ok=True)
Path("../figures").mkdir(parents=True, exist_ok=True)

scores_df.to_csv("../data/output/spot_celltype_scores.csv", index=False)
composition_df.to_csv("../data/output/spot_celltype_composition.csv", index=False)

save_celltype_composition_heatmap(
    composition_df,
    "../figures/celltype_composition_heatmap.png"
)

In [ ]:
The scoring results suggest that different spatial spots are preferentially aligned to different reference-derived cell-type signatures. After normalisation, these scores can be interpreted as relative composition estimates, providing a simplified view of how cell states may be distributed across spatial locations.